# Task 2 — Heritage Crack Detection

**Model:** YOLOv8s (pretrained COCO, fine-tuned)  
**Dataset:** OmniCrack30k — masks converted to bounding boxes  
**Target:** mAP@50 > 70%  

**Runtime required:** T4 GPU (`Runtime > Change runtime type > T4 GPU`)

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > T4 GPU"
print(f"GPU   : {torch.cuda.get_device_name(0)}")
print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
!pip install -q ultralytics

## 1. Mount Drive & Prepare Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/HeritagePreservation')
DATA_DIR  = Path('/content/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Drive contents:")
for f in sorted(DRIVE_DIR.iterdir()):
    print(f"  {f.name}")

In [ ]:
import zipfile

OMNI_DIR = DATA_DIR / 'omnicrack30k'

if OMNI_DIR.exists() and any(OMNI_DIR.rglob('*.jpg')):
    print("Already unzipped, skipping.")
else:
    zip_path = DRIVE_DIR / 'omnicrack30k.zip'
    size_gb = zip_path.stat().st_size / 1e9
    print(f"Unzipping {zip_path.name} ({size_gb:.2f} GB)...")
    print("Expected time: 10-20 minutes on Colab.")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(DATA_DIR)
    print("Done.")

In [ ]:
import os

# Auto-find the extracted folder (handles different zip structures)
OMNI_DIR = DATA_DIR / 'omnicrack30k'
if not OMNI_DIR.exists():
    candidates = [d for d in DATA_DIR.iterdir() if d.is_dir() and 'omni' in d.name.lower()]
    assert candidates, f"Cannot find omnicrack directory under {DATA_DIR}"
    OMNI_DIR = candidates[0]

# Find images and masks dirs
IMG_DIR  = OMNI_DIR / 'images'
MASK_DIR = OMNI_DIR / 'masks'

if not IMG_DIR.exists():
    for p in OMNI_DIR.rglob('images'):
        if p.is_dir():
            IMG_DIR  = p
            MASK_DIR = p.parent / 'masks'
            break

img_files  = sorted(IMG_DIR.glob('*.*'))
mask_files = sorted(MASK_DIR.glob('*.*'))
print(f"Dataset root : {OMNI_DIR}")
print(f"Images       : {len(img_files)}")
print(f"Masks        : {len(mask_files)}")
print(f"Sample imgs  : {[f.name for f in img_files[:3]]}")
print(f"Sample masks : {[f.name for f in mask_files[:3]]}")

## 2. Convert Masks → YOLO Labels

Each binary mask is converted to bounding boxes via connected components.  
Labels are written to `omnicrack30k/labels/` — same stem as the image.

In [ ]:
import cv2
import numpy as np


def mask_to_yolo_labels(mask_path, min_area=200):
    """Binary mask PNG → list of YOLO label strings '0 cx cy w h'."""
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []
    h, w = mask.shape
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    labels = []
    for cnt in contours:
        if cv2.contourArea(cnt) < min_area:
            continue
        x, y, bw, bh = cv2.boundingRect(cnt)
        cx = (x + bw / 2) / w
        cy = (y + bh / 2) / h
        nw = min(bw / w, 1.0)
        nh = min(bh / h, 1.0)
        labels.append(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
    return labels

print("Conversion function ready.")

In [ ]:
from tqdm.notebook import tqdm

LBL_DIR = OMNI_DIR / 'labels'

if LBL_DIR.exists() and len(list(LBL_DIR.glob('*.txt'))) > 1000:
    print(f"Labels already exist ({len(list(LBL_DIR.glob('*.txt')))}), skipping conversion.")
else:
    LBL_DIR.mkdir(exist_ok=True)
    print(f"Converting {len(img_files)} masks to YOLO labels...")
    no_crack = 0
    for img_path in tqdm(img_files):
        stem = img_path.stem
        mask_path = MASK_DIR / f"{stem}.png"
        if not mask_path.exists():
            mask_path = MASK_DIR / f"{stem}.bmp"
        labels = mask_to_yolo_labels(mask_path) if mask_path.exists() else []
        if not labels:
            no_crack += 1
        (LBL_DIR / f"{stem}.txt").write_text('\n'.join(labels))
    print(f"Done. Images with no crack boxes: {no_crack}/{len(img_files)}")

In [ ]:
from sklearn.model_selection import train_test_split

# 70 / 15 / 15 split
train_imgs, tmp_imgs = train_test_split(img_files, test_size=0.30, random_state=42)
val_imgs,  test_imgs = train_test_split(tmp_imgs,  test_size=0.50, random_state=42)

print(f"Train : {len(train_imgs)}")
print(f"Val   : {len(val_imgs)}")
print(f"Test  : {len(test_imgs)}")

# Write split text files (relative paths from OMNI_DIR)
def write_split(paths, out_path):
    lines = [str(p.relative_to(OMNI_DIR)) for p in paths]
    out_path.write_text('\n'.join(lines))

write_split(train_imgs, OMNI_DIR / 'train.txt')
write_split(val_imgs,   OMNI_DIR / 'val.txt')
write_split(test_imgs,  OMNI_DIR / 'test.txt')

# data.yaml
yaml_content = f"""path: {OMNI_DIR}
train: train.txt
val: val.txt
test: test.txt

nc: 1
names: ['crack']
"""
(OMNI_DIR / 'data.yaml').write_text(yaml_content)
print(f"data.yaml written to {OMNI_DIR / 'data.yaml'}")

## 3. Train YOLOv8

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')  # YOLOv8 small — good speed/accuracy tradeoff on T4

results = model.train(
    data=str(OMNI_DIR / 'data.yaml'),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    project='/content/runs',
    name='crack_detection',
    exist_ok=True,
    patience=10,       # early stopping
    save=True,
    plots=True,
    seed=42,
    amp=True,
)

## 4. Evaluation

In [ ]:
best_model = YOLO('/content/runs/crack_detection/weights/best.pt')

metrics = best_model.val(
    data=str(OMNI_DIR / 'data.yaml'),
    split='test',
    device=0,
    imgsz=640,
)

print("\n=== Test Results ===")
print(f"mAP@50     : {metrics.box.map50:.4f}  (target >0.70)")
print(f"mAP@50:95  : {metrics.box.map:.4f}")
print(f"Precision  : {metrics.box.mp:.4f}")
print(f"Recall     : {metrics.box.mr:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

run_dir = Path('/content/runs/crack_detection')

# Show training curves saved by ultralytics
for plot_file in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    p = run_dir / plot_file
    if p.exists():
        img = mpimg.imread(str(p))
        plt.figure(figsize=(14, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.title(plot_file)
        plt.tight_layout()
        plt.show()

In [ ]:
# Visualize predictions on 4 random test images
import random
import cv2
import numpy as np

sample_imgs = random.sample(test_imgs, min(4, len(test_imgs)))
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('YOLOv8 Predictions — Test Samples')

for ax, img_path in zip(axes, sample_imgs):
    result = best_model.predict(str(img_path), conf=0.25, verbose=False)[0]
    annotated = result.plot()
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    ax.imshow(annotated_rgb)
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/predictions_sample.png', dpi=150)
plt.show()

## 5. Save to Drive

In [ ]:
import shutil

DRIVE_CKPT  = DRIVE_DIR / 'checkpoints' / 'detector'
DRIVE_PLOTS = DRIVE_DIR / 'plots' / 'detector'
DRIVE_CKPT.mkdir(parents=True,  exist_ok=True)
DRIVE_PLOTS.mkdir(parents=True, exist_ok=True)

# Save weights
for w in ['best.pt', 'last.pt']:
    src = run_dir / 'weights' / w
    if src.exists():
        shutil.copy2(src, DRIVE_CKPT / w)

# Save plots
for plot_file in run_dir.glob('*.png'):
    shutil.copy2(plot_file, DRIVE_PLOTS / plot_file.name)
for plot_file in run_dir.glob('*.jpg'):
    shutil.copy2(plot_file, DRIVE_PLOTS / plot_file.name)

# Save prediction sample
shutil.copy2('/content/predictions_sample.png', DRIVE_PLOTS / 'predictions_sample.png')

# Save metrics summary
import json
summary = {
    'map50':     float(metrics.box.map50),
    'map50_95':  float(metrics.box.map),
    'precision': float(metrics.box.mp),
    'recall':    float(metrics.box.mr),
}
(DRIVE_CKPT / 'metrics.json').write_text(json.dumps(summary, indent=2))

print("Saved to Drive:")
for f in sorted(DRIVE_CKPT.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size/1e6:.1f} MB)")
print(f"\nmAP@50 : {summary['map50']:.4f}  (target >0.70)")